In [4]:
import pandas as pd

# Load raw Kaggle data
articles = pd.read_csv('../data/raw/articles.csv')
customers = pd.read_csv('../data/raw/customers.csv')
transactions = pd.read_csv('../data/raw/transactions_train.csv')

In [12]:
articles.head(3)

,article_id,product_code,prod_name,product_type_no,product_type_name,product_group_name,graphical_appearance_no,graphical_appearance_name,colour_group_code,colour_group_name,...,department_name,index_code,index_name,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name,detail_desc
0,108775015,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,9,Black,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
1,108775044,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,10,White,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
2,108775051,108775,Strap top (1),253,Vest top,Garment Upper body,1010017,Stripe,11,Off White,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.


In [24]:
articles.iloc[345]["detail_desc"]

'Cable-knit hat with a faux fur pompom.'

In [19]:
customers.head(3)

,customer_id,FN,Active,club_member_status,fashion_news_frequency,age,postal_code
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,NaN,NaN,ACTIVE,NONE,49.0,52043ee2162cf5aa7ee79974281641c6f11a68d276429a...
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,NaN,NaN,ACTIVE,NONE,25.0,2973abc54daa8a5f8ccfe9362140c63247c5eee03f1d93...
2,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,NaN,NaN,ACTIVE,NONE,24.0,64f17e6a330a85798e4998f62d0930d14db8db1c054af6...


In [9]:
transactions.head(3)

,t_dat,customer_id,article_id,price,sales_channel_id
0,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,663713001,0.050831,2
1,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,541518023,0.030492,2
2,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,505221004,0.015237,2


In [13]:
# 1. Create a clean string description per article
articles['item_text'] = (
    articles['prod_name'].fillna('') + " (" +
    articles['product_type_name'].fillna('') + ", " +
    articles['colour_group_name'].fillna('') + ", " +
    articles['perceived_colour_value_name'].fillna('') + ")"
)

article_text_map = dict(zip(articles['article_id'], articles['item_text']))

# 2. Filter to recent transactions (e.g., last 4 weeks) to keep it focused
recent_transactions = transactions[transactions['t_dat'] >= '2020-08-20'].copy()
recent_transactions.sort_values(by=['customer_id', 't_dat'], inplace=True)

# 3. Aggregate history per customer (max last 8 items to fit token limits)
grouped = (
    recent_transactions.groupby('customer_id')['article_id']
    .apply(list)
    .reset_index()
)

# Filter users who have at least 2 interactions (1 context + 1 target)
grouped = grouped[grouped['article_id'].apply(len) >= 2]

# 4. Format a sample prompt and target
sample_row = grouped.iloc[0]
history_ids = sample_row['article_id'][:-1][-8:]  # Up to last 8 context items
target_id = sample_row['article_id'][-1]          # Next item to predict

history_texts = " -> ".join([article_text_map.get(i, "Unknown Item") for i in history_ids])
prompt = f"Customer purchase history: {history_texts} -> Next purchase:"

print("PROMPT:")
print(prompt)
print("\nTARGET ITEM ID:")
print(target_id)
print("TARGET DESCRIPTION:")
print(article_text_map.get(target_id))

PROMPT:
Customer purchase history: Maja HW  WovenTRS (Trousers, Black, Dark) -> Next purchase:

TARGET ITEM ID:
448509014
TARGET DESCRIPTION:
Perrie Slim Mom Denim TRS (Trousers, Blue, Light)


In [23]:
# 1. Enhance Item Text with 'detail_desc' (Extract first sentence cleanly)
def get_first_sentence(text):
    text = str(text)
    if pd.isna(text) or not text.strip():
        return ""
    # Split by the first period, keep it, and strip extra whitespace
    return text.split('.')[0].strip() + "." if '.' in text else text.strip()

articles['detail_desc'] = articles['detail_desc'].apply(get_first_sentence)

articles['item_text'] = (
    articles['prod_name'].fillna('') + " (" +
    articles['product_type_name'].fillna('') + ", " +
    articles['colour_group_name'].fillna('') + ") - " +
    articles['detail_desc']
)
article_text_map = dict(zip(articles['article_id'], articles['item_text']))

# 2. Process Customer Demographics
customers['age'] = customers['age'].fillna(0).astype(int)
customers['club_member_status'] = customers['club_member_status'].fillna('UNKNOWN')
customer_profile_map = dict(
    zip(customers['customer_id'], 
        "Age: " + customers['age'].astype(str) + ", mem_status: " + customers['club_member_status'])
)

# 3. Format the Enhanced Prompt
sample_row = grouped.iloc[345]
cid = sample_row['customer_id']
history_ids = sample_row['article_id'][:-1][-5:]  # Reduced to last 5 items to fit richer text
target_id = sample_row['article_id'][-1]

user_profile = customer_profile_map.get(cid, "Age: Unknown, Status: Unknown")
history_texts = " \n-> ".join([article_text_map.get(i, "Unknown Item") for i in history_ids])

prompt = f"Customer Profile: [{user_profile}]\nPurchase History:\n-> {history_texts}\nNext purchase:"

print("ENHANCED PROMPT:\n")
print(prompt)

ENHANCED PROMPT:

Customer Profile: [Age: 50, mem_status: ACTIVE]
Purchase History:
-> Boye high shaft boot (Boots, Black) - Boots in imitation leather with sturdy shafts and a loop at the sides. 
-> Scout (Top, Beige) - Short dress in a soft rib knit containing some wool. 
-> Scooter (Sweater, Black) - Knee-length dress in a soft knit containing some wool with a ribbed polo neck, d 
-> Shanghai dress (Dress, Black) - Short dress in soft jersey made from a viscose blend with a round neckline, long 
-> Elle Sweater Dress (Dress, Black) - Short dress in sweatshirt fabric made from a cotton and recycled polyester blend
Next purchase:


In [2]:
import pandas as pd

# Load the generated sample
df = pd.read_parquet('../data/processed/train_sample.parquet')

# View the columns and the first sequence
print(f"Dataset shape: {df.shape}")
display(df.head(2))

# Print a formatted example of what the LLM will actually see
# Print a formatted example of what the LLM will actually see
row = df.iloc[-1] 
print("\n--- LLM INPUT PROMPT ---")
print(f"Customer Profile: [{row['user_profile']}]")
print("Purchase History:")
# No for-loop needed anymore, it's already perfectly formatted!
print(f"-> {row['context_texts']}")
print("Next purchase:")

print("\n--- TARGET (GROUND TRUTH) ---")
print(row['target_text'])

Dataset shape: (1851, 6)


,customer_id,context_ids,target_id,user_profile,context_texts,target_text
0,012032266bd72618a1bb05e741b2d6e3b61d950905a685...,0891591001,0884319001,"Age: 62, Status: ACTIVE, News Frequency: NONE","Lilly long shacket (Jacket, Check pattern, Dar...","Lucien CONSCIOUS (Blouse, All over pattern pat..."
1,012032266bd72618a1bb05e741b2d6e3b61d950905a685...,"0891591001,0884319001",0856808001,"Age: 62, Status: ACTIVE, News Frequency: NONE","Lilly long shacket (Jacket, Check pattern, Dar...","Uma (Dress, Solid pattern, Greenish Khaki, Dre..."



--- LLM INPUT PROMPT ---
Customer Profile: [Age: 19, Status: ACTIVE, News Frequency: Regularly]
Purchase History:
-> Speedy conscious tee (T-shirt, Solid pattern, Green, Ladies Sport Bras dept, Sport) - Straight-cut sports top in fast-drying mesh with short sleeves and a rounded hem.
-> FIG TANK (Vest top, Solid pattern, Black, Tops Woven dept, Divided) - Straight-cut top in an airy viscose weave with narrow, adjustable shoulder straps.
-> Tilda tank (Vest top, Melange pattern, Grey, Basic 1 dept, Divided) - Cropped, fitted top in cotton jersey with narrow shoulder straps.
-> Woody hoodie (Hoodie, Solid pattern, Black, Jersey Basic dept, Ladieswear) - Wide top in sweatshirt fabric with a lined drawstring hood, kangaroo pocket and ribbing at the cuffs and hem.
-> Tess Tee. (T-shirt, Solid pattern, Black, Basic 1 dept, Divided) - Long T-shirt in soft jersey made from a cotton blend with short sleeves with sewn-in turn-ups.
Next purchase:

--- TARGET (GROUND TRUTH) ---
2 Pack Karin Mid-S